In [ ]:
from pathlib import Path
from urllib.request import urlopen

import numpy as np
from alkaid.codegen import RTLModel
from alkaid.converter import trace_model
from alkaid.trace import FVArray, trace
from fqtree import FQTreeClassifier
from sklearn.datasets import fetch_openml

data = fetch_openml('UNSW_NB15', version=1)

if not Path('/tmp/unsw_nb15_binarized.npz').exists():
    print('Downloading preprocessed UNSW-NB15 dataset...')
    url = 'https://zenodo.org/record/4519767/files/unsw_nb15_binarized.npz?download=1'
    with urlopen(url) as response:
        if response.status != 200:
            raise RuntimeError(f'Failed to download dataset: {response.status} {response.reason}')
        with open('/tmp/unsw_nb15_binarized.npz', 'wb') as f:
            f.write(response.read())

unsw_nb15_data = np.load('/tmp/unsw_nb15_binarized.npz')
_train = unsw_nb15_data['train']
_test = unsw_nb15_data['test']
X_train, y_train = _train[:, :-1], _train[:, -1]
X_test, y_test = _test[:, :-1], _test[:, -1]

In [ ]:
def run(n_estimators, max_depth, scale, bias):
    model_large = FQTreeClassifier(
        scale=scale, bias=bias,
        num_class=1, n_estimators=n_estimators, max_depth=max_depth, eta=0.8, scale_pos_weight=0.15,
        objective='binary:logitraw'
    )
    model_large.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    print(f'Test acc: {np.mean(model_large.predict(X_test) == y_test):.4f}')

    hw_bst = model_large.ibooster()
    inp = FVArray.new(593).quantize(0, 1, 0).as_new()
    _, out = trace_model(hw_bst, inputs=inp, mode='mux')
    comb = trace(inp, out)

    train_acc = np.mean((comb.predict(X_train).ravel() >= 0) == y_train)
    test_acc = np.mean((comb.predict(X_test).ravel() >= 0) == y_test)
    print(f'HW Train acc: {train_acc:.4f}, Test acc: {test_acc:.4f}')
    # print(f'Estimated LUT: {comb.cost:.1f}')

    rtl = RTLModel(comb, f'/tmp/fqtree/nid-{n_estimators=}-{max_depth=}', 'model', n_stages=1, clock_period=1.5, clock_uncertainty=0, part_name='xcvu9p-flgb2104-2-i')
    rtl.write(xls_opt=True, metadata={'comb_metric': test_acc})

    rtl.write(xls_opt=True, metadata={'comb_metric': test_acc})
    for _ in range(4):
        try:
            rtl._compile(_env={'VERILATOR_FLAGS': ''}, nproc=4)
            break
        except Exception as _e:
            pass # verilator internal error
    else:
        raise RuntimeError('Failed to compile RTL model after 4 attempts')
    assert np.all(rtl.predict(X_test) == comb.predict(X_test)) # bit-exact check

In [3]:
run(8, 6, 2.0, -1.5)

Test acc: 0.9235
HW Train acc: 0.9223, Test acc: 0.9313


In [4]:
run(4, 6, 2.0, -1.5)

Test acc: 0.9119
HW Train acc: 0.9168, Test acc: 0.9284


In [5]:
run(2, 6, 2.0, -1.5)

Test acc: 0.9063
HW Train acc: 0.9040, Test acc: 0.9174
